In [1]:
import numpy as np
import onnx
import onnxruntime as rt
from onnxconverter_common import float16
import time

In [2]:
def benchmark(ort_sess, dtype='float32', n=100):   
    input = np.random.rand(1,3,320,320).astype(dtype)
    
    # initial run can vary a bit, so don't use it in the benchmark
    output = ort_sess.run(['output'], {'input': input})
    
    start = time.time()
    
    for _ in range(n):
        _ = ort_sess.run(['output'], {'input': input})
    
    end = time.time()
    
    return (end - start) * 1000 / n # average time in ms

In [16]:
#model_path = "../models/voc_pretrained_bnopt.onnx"
#model_path = "../models/person_only_both_datasets_4_layers_finetuned/model_best_no_bnopt.onnx"
model_path = "../models/person_only_both_datasets_4_layers_finetuned/model_best_bnopt.onnx"
#model_path = "../models/person_only_both_datasets_4_layers_finetuned/iterative_pruning_15_epochs_onestep/model_pruned_8.pt"

In [17]:
## load onnx model
dtype = 'float32'
model = onnx.load(model_path)
onnx.checker.check_model(model)
# create rt session
providers = ['CUDAExecutionProvider']#,'CPUExecutionProvider']
#providers = [('CUDAExecutionProvider', {"cudnn_conv_use_max_workspace": '1','cudnn_conv_algo_search': 'EXHAUSTIVE'}),'CPUExecutionProvider',]


## Graph Optimization

sess_options = rt.SessionOptions()
#sess_options.enable_profiling = True
#sess_options.graph_optimization_level = rt.GraphOptimizationLevel.ORT_ENABLE_EXTENDED
#sess_options.optimized_model_filepath = 'graph_opt_model.onnx'

## FP16

#model_path = 'model_fp16.onnx'
#model_fp16 = float16.convert_float_to_float16(model)
#onnx.save(model_fp16, model_path)
#dtype = 'float16'

## create session
ort_sess = rt.InferenceSession(model_path, sess_options=sess_options, providers=providers)


In [22]:
total_time = benchmark(ort_sess, dtype)
print(f'total time = {total_time:.1f}ms')

total time = 49.3ms
